In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = (
    Path.home()
    / "Downloads"
    / "IDX-Exchange-Data-Science"
)

# Load the cleaned datasets.
train_df = pd.read_csv(
    PROJECT_DIR / "single_family_train_cleaned.csv",
    dtype={"PostalCode": "string"},
    low_memory=False,
)

test_df = pd.read_csv(
    PROJECT_DIR / "single_family_test_cleaned.csv",
    dtype={"PostalCode": "string"},
    low_memory=False,
)

# Separate the target and exclude sale dates from model inputs.
y_train = pd.to_numeric(train_df["ClosePrice"], errors="raise")
y_test = pd.to_numeric(test_df["ClosePrice"], errors="raise")

X_train = train_df.drop(columns=["ClosePrice", "CloseDate"]).copy()
X_test = test_df.drop(columns=["ClosePrice", "CloseDate"]).copy()

# Verify feature consistency and positive, finite targets.
assert X_train.columns.equals(X_test.columns)
assert np.isfinite(y_train).all() and y_train.gt(0).all()
assert np.isfinite(y_test).all() and y_test.gt(0).all()

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training missing feature values:", int(X_train.isna().sum().sum()))
print("Test missing feature values:", int(X_test.isna().sum().sum()))

Training features: (278710, 24)
Test features: (11684, 24)
Training missing feature values: 3
Test missing feature values: 0


In [2]:
# Identify columns with missing values.
missing_counts = X_train.isna().sum()

display(
    missing_counts[missing_counts.gt(0)]
    .rename("Missing count")
    .to_frame()
)

# Inspect affected records.
display(
    X_train.loc[X_train.isna().any(axis=1)]
)

,Missing count
SubdivisionName,3


,Flooring,ViewYN,PoolPrivateYN,Latitude,Longitude,LivingArea,MLSAreaMajor,CountyOrParish,AttachedGarageYN,ParkingTotal,...,BedroomsTotal,FireplaceYN,Stories,Levels,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,LotSizeSquareFeet
130672,unknown,false,false,33.659484,-117.998803,1252.0,17 - northwest huntington beach,orange,true,2.0,...,3.0,false,1.0,one,3.0,false,2.0,huntington beach union high,92647,5800.0
154626,unknown,true,false,33.935011,-116.983153,999.0,263 - banning/beaumont/cherry valley,riverside,true,1.0,...,3.0,false,1.0,one,3.0,false,1.0,beaumont,92223,9583.0
223909,unknown,false,false,33.624026,-117.685913,2518.0,ls - lake forest south,orange,true,3.0,...,4.0,true,2.0,two,1.0,false,3.0,saddleback valley unified,92630,4982.0


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numeric features.
categorical_columns = X_train.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

numeric_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]

# Standardize category labels and fill missing categories.
for dataset in [X_train, X_test]:
    for col in categorical_columns:
        dataset[col] = (
            dataset[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
            .fillna("unknown")
        )

assert not X_train.isna().any().any()
assert not X_test.isna().any().any()

# Encode categories and standardize numeric features.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_columns,
        ),
        ("numeric", StandardScaler(), numeric_columns),
    ],
    sparse_threshold=1.0,
)

# Learn preprocessing rules from training data only.
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Training matrix:", X_train_ready.shape)
print("Test matrix:", X_test_ready.shape)

Training matrix: (278710, 16661)
Test matrix: (11684, 16661)


In [4]:
import joblib

# Save prepared features, original-price targets, and preprocessing rules.
prepared_data = {
    "X_train_ready": X_train_ready,
    "X_test_ready": X_test_ready,
    "y_train": y_train,
    "y_test": y_test,
    "preprocessor": preprocessor,
    "feature_names": preprocessor.get_feature_names_out(),
}

prepared_path = PROJECT_DIR / "prepared_modeling_data.joblib"

joblib.dump(prepared_data, prepared_path, compress=3)

print("Prepared modeling data saved:", prepared_path)

Prepared modeling data saved: /Users/jasmineyuan/Downloads/IDX-Exchange-Data-Science/prepared_modeling_data.joblib


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

# Train linear regression on original sale prices.
raw_model = LinearRegression()
raw_model.fit(X_train_ready, y_train)

# Train linear regression on log-transformed sale prices.
log_model = LinearRegression()
log_model.fit(X_train_ready, np.log(y_train))

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](16661,)","[0.01,0.26,0.03,...,0.01,0.02,0.03]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,13.33
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16661


In [6]:
# Inspect test prices and the largest dollar errors.
test_audit = test_df[
    ["ClosePrice", "LivingArea", "City", "PostalCode"]
].copy()

test_audit["PredictedPrice"] = np.exp(
    log_model.predict(X_test_ready)
)
test_audit["AbsoluteError"] = (
    test_audit["ClosePrice"] - test_audit["PredictedPrice"]
).abs()

display(
    test_audit.nlargest(10, "AbsoluteError")
)

,ClosePrice,LivingArea,City,PostalCode,PredictedPrice,AbsoluteError
11599,31750000.0,4351.0,beverly hills,90210,9.760129e+06,2.198987e+07
11130,19750000.0,3300.0,laguna beach,92651,7.326296e+06,1.242370e+07
9143,19995000.0,2502.0,malibu,90265,9.603587e+06,1.039141e+07
8593,11150000.0,2219.0,encinitas,92024,2.040804e+06,9.109196e+06
495,9750000.0,2631.0,carlsbad,92008,1.989871e+06,7.760129e+06
11220,11050000.0,3209.0,del mar,92014,4.087788e+06,6.962212e+06
3252,8900000.0,3210.0,woodside,94062,3.238372e+06,5.661628e+06
266,9950000.0,3463.0,san francisco,94121,4.368245e+06,5.581755e+06
8580,11150000.0,5442.0,beverly hills,90210,5.769068e+06,5.380932e+06
9416,13000000.0,4492.0,hermosa beach,90254,7.659516e+06,5.340484e+06


In [7]:
# Find the two suspect sale prices in the original CSV files.
RAW_DIR = Path.home() / "Downloads" / "idx exchange" / "csv"

columns_to_check = [
    "ListingKey",
    "ListingId",
    "UnparsedAddress",
    "City",
    "PostalCode",
    "PropertyType",
    "PropertySubType",
    "LivingArea",
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "CloseDate",
]

matches = []
files = sorted(RAW_DIR.glob("CRMLSSold*.csv"))

if not files:
    raise FileNotFoundError(f"No matching CSV files found in {RAW_DIR}")

for file in files:
    for chunk in pd.read_csv(
        file,
        usecols=lambda col: col in columns_to_check,
        dtype="string",
        chunksize=50000,
    ):
        prices = pd.to_numeric(
            chunk["ClosePrice"].str.replace(",", "", regex=False)
                               .str.replace("$", "", regex=False),
            errors="coerce",
        )

        matched = chunk.loc[
            prices.isin([796000000, 645000000])
        ].copy()

        if not matched.empty:
            matched.insert(0, "SourceFile", file.name)
            matches.append(matched)

if matches:
    original_records = pd.concat(matches, ignore_index=True)

    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
    ):
        display(original_records)
else:
    print("No matching records found.")

,SourceFile,OriginalListPrice,ListingKey,CloseDate,ClosePrice,UnparsedAddress,PropertyType,LivingArea,ListPrice,PropertySubType,ListingId,City,PostalCode
0,CRMLSSold202604.csv,645000.0,1153305774,2026-04-22,645000000.0,3721 Magellan Circle,Residential,1104.0,645000.0,SingleFamilyResidence,NDP2601657,Oceanside,92056
1,CRMLSSold202604.csv,849900.0,1149697906,2026-04-27,796000000.0,9168 Old Castle Road Road,Residential,1936.0,824900.0,SingleFamilyResidence,NDP2511562,Valley Center,92082


In [8]:
# Evaluate the log-price model in dollars.
log_predictions = np.exp(
    log_model.predict(X_test_ready)
)

log_metrics_df = pd.DataFrame({
    f"Log model — Test metrics (n = {len(y_test):,})": [
        "R² (price)",
        "MAE",
        "RMSE",
        "MAPE",
    ],
    "Value": [
        f"{r2_score(y_test, log_predictions):.4f}",
        f"${mean_absolute_error(y_test, log_predictions):,.2f}",
        f"${np.sqrt(mean_squared_error(y_test, log_predictions)):,.2f}",
        f"{mean_absolute_percentage_error(y_test, log_predictions) * 100:.2f}%",
    ],
})

display(log_metrics_df.style.hide(axis="index"))

"Log model — Test metrics (n = 11,684)",Value
R² (price),0.8256
MAE,"$172,630.73"
RMSE,"$455,641.17"
MAPE,12.45%


In [9]:
# Evaluate the original-price model in dollars.
raw_predictions = raw_model.predict(X_test_ready)

raw_metrics_df = pd.DataFrame({
    f"Original model — Test metrics (n = {len(y_test):,})": [
        "R² (price)",
        "MAE",
        "RMSE",
        "MAPE",
    ],
    "Value": [
        f"{r2_score(y_test, raw_predictions):.4f}",
        f"${mean_absolute_error(y_test, raw_predictions):,.2f}",
        f"${np.sqrt(mean_squared_error(y_test, raw_predictions)):,.2f}",
        f"{mean_absolute_percentage_error(y_test, raw_predictions) * 100:.2f}%",
    ],
})

display(raw_metrics_df.style.hide(axis="index"))

"Original model — Test metrics (n = 11,684)",Value
R² (price),0.7770
MAE,"$231,932.38"
RMSE,"$515,209.13"
MAPE,21.54%


- **Baseline improvement:** Test price-scale R² increased from approximately 0.01 to 0.83 after correcting two erroneous sale prices, with no change to the log-linear model or removal of test records.
- **Data quality investigation:** Inspected the largest prediction errors and traced two suspicious prices to the original CSV records. The recorded prices were 1,000 times the checked sale prices and accounted for 99.77% of the original test squared error. Corrections were incorporated into notebook 02.
- **Log-linear regression:** Trained linear regression on `log(ClosePrice)` and converted predictions back to dollars for evaluation. On the full corrected test set, the log model achieved R² = 0.8256 and MAPE = 12.45%, compared with R² = 0.7244 and MAPE = 24.17% for the original-price model.